In [1]:
import numpy as np
import math
from sympy import *
from scipy import integrate
import matplotlib.pyplot as plt
import random
import time

In [2]:
def TDT(b,c):
    if b+c == 0:
        return 0
    else:
        return ((b-c)**2)/(b+c)

In [3]:
def LS(b,c,N): 
    stat = TDT(b,c)
    M = stat; m = stat
    
    if b >= 2:
        v = [TDT(b-2,c), TDT(b-2,c+1), TDT(b-2,c+2)]
        M = max([max(v),M])
        m = min([min(v),m])
    if b >= 1:
        v = [TDT(b-1,c), TDT(b-1,c+1)]
        M = max([max(v),M])
        m = min([min(v),m])
    if b >= 1 and c >= 1:
        v = TDT(b-1,c-1)
        M = max([v,M])
        m = min([v,m])
    if c >= 2:
        v = [TDT(b,c-2), TDT(b+1,c-2), TDT(b+2,c-2)]
        M = max([max(v),M])
        m = min([min(v),m])
    if c >= 1:
        v = [TDT(b,c-1), TDT(b+1,c-1)]
        M = max([max(v),M])
        m = min([min(v),m])
    if b+c <= 2*N-2:
        v = [TDT(b+1,c+1), TDT(b,c+2), TDT(b+2,c)]
        M = max([max(v),M])
        m = min([min(v),m])
    if b+c <= 2*N-1:
        v = [TDT(b,c+1), TDT(b+1,c)]
        M = max([max(v),M])
        m = min([min(v),m])
    if b+c <= 2*N-1 and b >= 1:
        v = TDT(b-1,c+2)
        M = max([v,M])
        m = min([v,m])
    if b+c <= 2*N-1 and c >= 1:
        v = TDT(b+2,c-1)
        M = max([v,M])
        m = min([v,m])
    
    return [M-stat, stat-m]

In [4]:
def d(x,b,c):
    if b >= x[0] and c >= x[1]:
        return math.ceil(((b+c)-(x[0]+x[1]))/2)
    elif b <= x[0] and c <= x[1]:
        return math.ceil(((x[0]+x[1])-(b+c))/2)
    else:
        return math.ceil(max([math.fabs(b-x[0]), math.fabs(c-x[1])])/2)

def ud(x,N): 
    sd = 10000
    for c in range(int(N/4)):
        s = 9+7*c
        for b in range(s,2*N-c+1):
            sd = min([d(x,b,c),d(x,b-2,c+2),sd])
    for b in range(int(N/4)):
        s = 9+7*b
        for c in range(s,2*N-b+1):
            sd = min([d(x,b,c),d(x,b+2,c-2),sd])
    return sd

In [5]:
def LP(x,eps,N,m): #M_{LP}
    u = np.zeros(m); v = np.zeros(m)
    scale = (8*(N-1)/N)/eps
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        v[i] = u[i] + np.random.laplace(0,2*scale)
    return np.argmax(v)

def ELP(x,eps,N,m): #M_{ELP}
    u = np.zeros(m); v = np.zeros(m)
    scale = 2*(8*(N-1)/N)/eps
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        v[i] = u[i] + np.random.laplace(0,scale)
    return np.argmax(v)

def EDLP(x,eps,N,m): #M_{EDLP}
    u = np.zeros(m); v = np.zeros(m)
    scale = 2*(8*(N-1)/N)/eps
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        v[i] = u[i] + np.random.laplace(0,scale)
    return np.argmax(v)

In [6]:
def h4(z):
    return np.sqrt(2)/(math.pi*(z**4+1))

def cdf(x,h):
    return integrate.quad(h,-oo,x)

def noise(h,CDF):
    rr = np.random.rand()
    l = 0; r = 2000
    while(1):
        t = (int)((l+r)/2)
        if CDF[t] >= rr:
            r = t
        else:
            l = t
        if r-l <= 1:
            z = (((l+r)/2)-1000)/100
            break
    return z

def SPS(x,eps,N,m,gamma,CDF): #M_{SPS}, ud(x), Two-Sided noise #N = 215, gamma = 4
    GS = 8*(N-1)/N; lbeta = math.log(GS/max(LS(131,299,N)))/39
    alpha = eps/(2*((gamma-1)**((gamma-1)/gamma)))
    beta = eps/(2*(gamma-1)); lbeta = min(lbeta,beta/m)
    k = 1 - m*lbeta/(2*beta)
    u = np.zeros(m); v = np.zeros(m)
    s = np.zeros(m)
    
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        if max(LS(x[i][0],x[i][1],N)) > 6:
            s[i] = GS
        else:
            s[i] = GS*math.exp(-lbeta*ud(x[i],N))
            
    S = max(s)
    for i in range(m):
        v[i] = u[i] + (S/(k*alpha))*noise(h4,CDF)
    
    return np.argmax(v)

def ESPS(x,eps,N,m,gamma,CDF): #M_{ESPS}, ud(x), Two-Sided noise #N=215, gamma = 4
    GS = 2*(8*(N-1)/N); lbeta = math.log(GS/(max(LS(75,355,N))+max(LS(182,246,N))))/39 
    alpha = eps/(2*((gamma-1)**((gamma-1)/gamma)))
    beta = eps/(2*(gamma-1)); lbeta = min(lbeta,beta/m);
    k = 1 - m*lbeta/(2*beta)
    u = np.zeros(m); v = np.zeros(m)
    s = np.zeros(m)
    
    flag = 0; sd = 10000
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        b1 = x[i][0]; c1 = x[i][1]
        ls1 = max(LS(b1,c1,N))
        for j in range(i+1,m):
            b2 = x[j][0]; c2 = x[j][1]
            ls2 = max(LS(b2,c2,N))
            if ls1 + ls2 > 12:
                flag = 1; break
            else:
                sd = min([math.ceil((12-ls1-ls2)*(b1+c1)*(b2+c2)/(32*(b1+c1+b2+c2)))+1, sd])
            
    if flag == 1:
        S = GS
    else:
        S = GS*math.exp(-lbeta*sd)
            
    for i in range(m):
        v[i] = u[i] + (S/(2*k*alpha))*noise(h4,CDF)
    
    return np.argmax(v)

def EDSPS(x,eps,N,m,gamma,CDF): #M_{EDSPS}, ud(x), Two-Sided noise #N=215, gamma = 4
    GS = 2*(8*(N-1)/N); lbeta = math.log(GS/(LS(42,388,N)[0]+LS(214,216,N)[1]))/39
    alpha = eps/(2*((gamma-1)**((gamma-1)/gamma)))
    beta = eps/(2*(gamma-1)); lbeta = min(lbeta,beta/m);
    k = 1 - m*lbeta/(2*beta)
    u = np.zeros(m); v = np.zeros(m)
    s = np.zeros(m)
    
    flag = 0; sd = 10000
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        b1 = x[i][0]; c1 = x[i][1]
        ls1 = LS(b1,c1,N); ls10 = ls1[0]; ls11 = ls1[1]
        for j in range(i+1,m):
            b2 = x[j][0]; c2 = x[j][1]
            ls2 = LS(b2,c2,N); ls20 = ls2[0]; ls21 = ls2[1]
            if ls10 + ls21 > 12 or ls11 + ls20 > 12:
                flag = 1; break
            else:
                sd = min([math.ceil((12-ls10-ls21)*(b1+c1)*(b2+c2)/(32*(b1+c1+b2+c2)))+1, math.ceil((12-ls11-ls20)*(b1+c1)*(b2+c2)/(32*(b1+c1+b2+c2)))+1, sd])
                
    if flag == 1:
        S = GS
    else:
        S = GS*math.exp(-lbeta*sd)
            
    for i in range(m):
        v[i] = u[i] + (S/(2*k*alpha))*noise(h4,CDF)
    
    return np.argmax(v)

In [7]:
def EP(x,eps,N,m): #M_{EP}
    u = np.zeros(m); v = np.zeros(m)
    beta = (8*(N-1)/N)/eps
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        v[i] = u[i] + np.random.exponential(2*beta,1)
    return np.argmax(v)

def EDOEP(x,eps,N,m): #M_{EDOEP}
    u = np.zeros(m); v = np.zeros(m)
    beta = 2*(8*(N-1)/N)/eps
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        v[i] = u[i] + np.random.exponential(beta,1)
    return np.argmax(v)

In [8]:
def h4_OS(z):
    if z < 0:
        return 0
    else:
        return 2*np.sqrt(2)/(math.pi*(1+z**4))

def SPS_OS(x,eps,N,m,gamma,CDF): #M_{SPS}, ud(x), One-Sided noise #N = 215, gamma = 4
    GS = 8*(N-1)/N; lbeta = math.log(GS/max(LS(131,299,N)))/39
    alpha = eps/(2*((gamma-1)**((gamma-1)/gamma)))
    beta = eps/(2*(gamma-1)); lbeta = min(lbeta,beta/(m/(gamma-1))) #min(lbeta,beta/(m-1)) 
    #k = 1 - (m-1)*lbeta/(2*beta)
    k = 1 - m*lbeta/(2*(gamma-1)*beta)
    u = np.zeros(m); v = np.zeros(m)
    s = np.zeros(m)
    
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        if max(LS(x[i][0],x[i][1],N)) > 6:
            s[i] = GS
        else:
            s[i] = GS*math.exp(-lbeta*ud(x[i],N))
            
    S = max(s)
    for i in range(m):
        v[i] = u[i] + (S/(k*alpha))*noise(h4_OS,CDF)
    
    return np.argmax(v)

def EDOSPS(x,eps,N,m,gamma,CDF): #M_{EDOSPS}, ud(x), One-Sided noise #N = 215, gamma = 4
    GS = 2*(8*(N-1)/N); lbeta = math.log(GS/(LS(42,388,N)[0]+LS(214,216,N)[1]))/39
    alpha = eps/(2*((gamma-1)**((gamma-1)/gamma)))
    beta = eps/(2*(gamma-1)); lbeta = min(lbeta,beta/(m/(gamma-1)))
    k = 1 - m*lbeta/(2*(gamma-1)*beta)
    u = np.zeros(m); v = np.zeros(m)
    s = np.zeros(m)
    
    flag = 0; sd = 10000
    for i in range(m):
        u[i] = TDT(x[i][0],x[i][1])
        b1 = x[i][0]; c1 = x[i][1]
        ls1 = LS(b1,c1,N); ls10 = ls1[0]; ls11 = ls1[1]
        for j in range(i+1,m):
            b2 = x[j][0]; c2 = x[j][1]
            ls2 = LS(b2,c2,N); ls20 = ls2[0]; ls21 = ls2[1]
            if ls10 + ls21 > 12 or ls11 + ls20 > 12:
                flag = 1; break
            else:
                sd = min([math.ceil((12-ls10-ls21)*(b1+c1)*(b2+c2)/(32*(b1+c1+b2+c2)))+1, math.ceil((12-ls11-ls20)*(b1+c1)*(b2+c2)/(32*(b1+c1+b2+c2)))+1, sd])
                
    if flag == 1:
        S = GS
    else:
        S = GS*math.exp(-lbeta*sd)
            
    for i in range(m):
        v[i] = u[i] + (S/(2*k*alpha))*noise(h4_OS,CDF)
    
    return np.argmax(v)

In [9]:
m = 6; chi2 = np.zeros(m); x = np.zeros((m,2))

chi2[0:6] = [29.83, 29.96, 33.76, 31.74, 30.52, 32.67]
for i in range(m):
    while(1):
        x[i][1] = np.random.randint(0, 216)
        x[i][0] = int((2*x[i][1] + chi2[i] + math.sqrt(chi2[i]*(chi2[i] + 8*x[i][1]))) / 2)+1
        if(x[i][0] + x[i][1] <= 430):
            break
print(x)

[[211. 112.]
 [207. 109.]
 [256. 140.]
 [257. 144.]
 [137.  59.]
 [162.  74.]]


In [10]:
def accuracy(x,eps,N,m,CDF,CDF_OS):
    stats = np.zeros((m,2)); acc = np.zeros(10)
    
    for j in range(1000):
        for i in range(m):
            stats[i][0] = TDT(x[i][0],x[i][1])
            stats[i][1] = i
        sstats = sorted(stats,key=lambda x:(x[0]),reverse=True)
        sx = np.zeros((m,2))
        for i in range(m):
            sx[i] = x[int(sstats[i][1])]
            
        A = LP(sx,eps,N,m); B = ELP(sx,eps,N,m); C = EDLP(sx,eps,N,m)
        D = SPS(sx,eps,N,m,4,CDF); E = ESPS(sx,eps,N,m,4,CDF); F = EDSPS(sx,eps,N,m,4,CDF)
        G = EP(sx,eps,N,m); H = EDOEP(sx,eps,N,m)
        I = SPS_OS(sx,eps,N,m,4,CDF_OS); J = EDOSPS(sx,eps,N,m,4,CDF_OS)
        
        if A == 0:
            acc[0] += 1
        if B == 0:
            acc[1] += 1
        if C == 0:
            acc[2] += 1
        if D == 0:
            acc[3] += 1
        if E == 0:
            acc[4] += 1
        if F == 0:
            acc[5] += 1
        if G == 0:
            acc[6] += 1
        if H == 0:
            acc[7] += 1
        if I == 0:
            acc[8] += 1
        if J == 0:
            acc[9] += 1
        #print(acc)
        
    return acc/1000

In [11]:
N = 215; eps = 15

CDF = np.zeros(2001)
for i in range(2001):
    CDF[i] = cdf((i-1000)/100,h4)[0]
    
CDF_OS = np.zeros(2001)
for i in range(2001):
    CDF_OS[i] = cdf((i-1000)/100,h4_OS)[0]

print("LP, ELP, EDLP, SPS, ESPS, EDSPS, EP, EDOEP, SPS_OS, EDOSPS")
print(accuracy(x,eps,N,m,CDF,CDF_OS))

LP, ELP, EDLP, SPS, ESPS, EDSPS, EP, EDOEP, SPS_OS, EDOSPS
[0.663 0.65  0.637 0.608 0.658 0.664 0.734 0.725 0.76  0.795]
